In [12]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    PowerTransformer
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import joblib


In [2]:
df = pd.read_csv("../datasets/overcrowding_prediction_dataset.csv")
df.drop(columns=['Prison_ID'],inplace=True)

In [4]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Avoid division by zero
        capacity = X["Capacity"].replace(0, np.nan)
        occupancy_30 = X["Occupancy_30_Days_Ago"].replace(0, np.nan)

        X["Current_Occupancy_Rate"] = (
            X["Current_Occupancy"] / capacity
        ) * 100

        X["Average_Daily_Admissions"] = (
            X["Admissions_Last_30_Days"] / 30
        )

        X["Monthly_Growth_Rate"] = (
            (X["Current_Occupancy"] - occupancy_30)
            / occupancy_30
        ) * 100

        # Replace NaN/inf if they occurred
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(0)

        return X

In [5]:
categorical_features = [
    "Prison_Type",
    "Security_Level"
]

numeric_features = [
    "Capacity",
    "Number_of_Blocks",
    "Current_Occupancy",

    "Admissions_Last_7_Days",
    "Admissions_Last_30_Days",
    "Pending_Admissions",

    "Releases_Last_7_Days",
    "Releases_Last_30_Days",
    "Upcoming_Releases_30_Days",
    "Upcoming_Releases_60_Days",
    "Upcoming_Releases_90_Days",

    "Transfers_In_Last_30_Days",
    "Transfers_Out_Last_30_Days",
    "Pending_Transfers_In",
    "Pending_Transfers_Out",

    "Average_Remaining_Sentence",
    "Median_Remaining_Sentence",

    "Occupancy_7_Days_Ago",
    "Occupancy_30_Days_Ago",
    "Occupancy_60_Days_Ago",
    "Occupancy_90_Days_Ago",

    "Current_Occupancy_Rate",
    "Average_Daily_Admissions",
    "Monthly_Growth_Rate"
]

In [6]:
numeric_pipeline = Pipeline([
    ("yeojohnson", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [7]:
X = df.drop(columns=[
    "Occupancy_30_Days",
    "Occupancy_60_Days",
    "Occupancy_90_Days"
])
y = df[[
    "Occupancy_30_Days",
    "Occupancy_60_Days",
    "Occupancy_90_Days"
]]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
xgb_model = Pipeline([
    ("feature_engineering", FeatureEngineering()),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ))
])
xgb_model.fit(X_train, y_train)
xgb_model.score(X_test, y_test)


0.9984919428825378

In [9]:

sample = pd.DataFrame([{
    "Prison_Type": "Maximum Security",
    "Security_Level": "High",

    "Capacity": 1000,
    "Number_of_Blocks": 8,
    "Current_Occupancy": 920,

    "Admissions_Last_7_Days": 18,
    "Admissions_Last_30_Days": 75,
    "Pending_Admissions": 12,

    "Releases_Last_7_Days": 10,
    "Releases_Last_30_Days": 42,
    "Upcoming_Releases_30_Days": 35,
    "Upcoming_Releases_60_Days": 58,
    "Upcoming_Releases_90_Days": 80,

    "Transfers_In_Last_30_Days": 11,
    "Transfers_Out_Last_30_Days": 8,
    "Pending_Transfers_In": 5,
    "Pending_Transfers_Out": 3,

    "Average_Remaining_Sentence": 7.5,
    "Median_Remaining_Sentence": 6.8,

    "Occupancy_7_Days_Ago": 915,
    "Occupancy_30_Days_Ago": 900,
    "Occupancy_60_Days_Ago": 880,
    "Occupancy_90_Days_Ago": 860
}])
prediction = xgb_model.predict(sample)

print("Predicted Future Occupancy:", prediction[0])

Predicted Future Occupancy: [937.3021  958.1062  993.88153]


In [14]:
joblib.dump(
    xgb_model,
    "../../models/overcrowding.pkl"
)

['../../models/overcrowding.pkl']